# Chapter 10: Selecting and Creating Columns

In [1]:
import polars as pl
pl.__version__  # The book is built with Polars version 1.20.0

'2.0.0'

In [2]:
starwars = pl.read_parquet("data/starwars.parquet")
rebels = starwars.drop("films").filter(
    pl.col("name").is_in(["Luke Skywalker", "Leia Organa", "Han Solo"])
)

print(rebels[:, :6])  
print(rebels[:, 6:11])
print(rebels[:, 11:])

shape: (3, 6)
┌────────────────┬────────┬──────┬────────────┬────────────┬───────────┐
│ name           ┆ height ┆ mass ┆ hair_color ┆ skin_color ┆ eye_color │
│ ---            ┆ ---    ┆ ---  ┆ ---        ┆ ---        ┆ ---       │
│ str            ┆ u16    ┆ f64  ┆ str        ┆ str        ┆ str       │
╞════════════════╪════════╪══════╪════════════╪════════════╪═══════════╡
│ Han Solo       ┆ 180    ┆ 80.0 ┆ brown      ┆ fair       ┆ brown     │
│ Leia Organa    ┆ 150    ┆ 49.0 ┆ brown      ┆ light      ┆ brown     │
│ Luke Skywalker ┆ 172    ┆ 77.0 ┆ blond      ┆ fair       ┆ blue      │
└────────────────┴────────┴──────┴────────────┴────────────┴───────────┘
shape: (3, 5)
┌────────────┬────────┬───────────┬───────────┬─────────┐
│ birth_year ┆ sex    ┆ gender    ┆ homeworld ┆ species │
│ ---        ┆ ---    ┆ ---       ┆ ---       ┆ ---     │
│ f64        ┆ cat    ┆ cat       ┆ str       ┆ str     │
╞════════════╪════════╪═══════════╪═══════════╪═════════╡
│ 29.0       ┆ male   ┆ m

In [6]:
print(starwars.glimpse(max_items_per_column=2))

Rows: 87
Columns: 16
$ name                 <str> 'Ackbar', 'Adi Gallia'
$ height               <u16> 180, 184
$ mass                 <f64> 83.0, 50.0
$ hair_color           <str> null, null
$ skin_color           <str> 'brown mottle', 'dark'
$ eye_color            <str> 'orange', 'blue'
$ birth_year           <f64> 41.0, null
$ sex                  <cat> male, female
$ gender               <cat> masculine, feminine
$ homeworld            <str> 'Mon Cala', 'Coruscant'
$ species              <str> 'Mon Calamari', 'Tholothian'
$ films          <list[str]> ['Return of the Jedi', 'The Force Awakens'], ['The Phantom Menace', 'Revenge of the Sith']
$ vehicles       <list[str]> null, null
$ starships      <list[str]> null, null
$ birth_date          <date> 1936-06-04, null
$ screen_time <duration[μs]> null, null

None


## Selecting Columns

In [7]:
rebels.select(
    "name",
    pl.col("homeworld"),
    pl.col("^.*_color$"),
    (pl.col("height") / 100).alias("height_m"),
)

name,homeworld,hair_color,skin_color,eye_color,height_m
str,str,str,str,str,f64
"""Han Solo""","""Corellia""","""brown""","""fair""","""brown""",1.8
"""Leia Organa""","""Alderaan""","""brown""","""light""","""brown""",1.5
"""Luke Skywalker""","""Tatooine""","""blond""","""fair""","""blue""",1.72


### Introducing Selectors

In [8]:
import polars.selectors as cs

In [10]:
rebels

name,height,mass,hair_color,skin_color,eye_color,birth_year,sex,gender,homeworld,species,vehicles,starships,birth_date,screen_time
str,u16,f64,str,str,str,f64,cat,cat,str,str,list[str],list[str],date,duration[μs]
"""Han Solo""",180,80.0,"""brown""","""fair""","""brown""",29.0,"""male""","""masculine""","""Corellia""","""Human""",null,"[""Millennium Falcon"", ""Imperial shuttle""]",1948-06-01,1h 12m 37s
"""Leia Organa""",150,49.0,"""brown""","""light""","""brown""",19.0,"""female""","""feminine""","""Alderaan""","""Human""","[""Imperial Speeder Bike""]",null,1958-05-30,1h 3m 40s
"""Luke Skywalker""",172,77.0,"""blond""","""fair""","""blue""",19.0,"""male""","""masculine""","""Tatooine""","""Human""","[""Snowspeeder"", ""Imperial Speeder Bike""]","[""X-wing"", ""Imperial shuttle""]",1958-05-30,1h 58m 44s


In [18]:
rebels.select(
    "name",
    cs.by_name("homeworld"),
    cs.matches("^.*_color$"),
    (cs.by_name("height") / 100).alias("height_m"),
    cs.ends_with("_color").name.suffix('@new')
)

name,homeworld,hair_color,skin_color,eye_color,height_m,hair_color@new,skin_color@new,eye_color@new
str,str,str,str,str,f64,str,str,str
"""Han Solo""","""Corellia""","""brown""","""fair""","""brown""",1.8,"""brown""","""fair""","""brown"""
"""Leia Organa""","""Alderaan""","""brown""","""light""","""brown""",1.5,"""brown""","""light""","""brown"""
"""Luke Skywalker""","""Tatooine""","""blond""","""fair""","""blue""",1.72,"""blond""","""fair""","""blue"""


### Selecting Based on Name

In [19]:
rebels.select(cs.starts_with("birth_"))

birth_year,birth_date
f64,date
29.0,1948-06-01
19.0,1958-05-30
19.0,1958-05-30


In [20]:
rebels.select(cs.ends_with("_color"))

hair_color,skin_color,eye_color
str,str,str
"""brown""","""fair""","""brown"""
"""brown""","""light""","""brown"""
"""blond""","""fair""","""blue"""


In [21]:
rebels.select(cs.contains("_"))

hair_color,skin_color,eye_color,birth_year,birth_date,screen_time
str,str,str,f64,date,duration[μs]
"""brown""","""fair""","""brown""",29.0,1948-06-01,1h 12m 37s
"""brown""","""light""","""brown""",19.0,1958-05-30,1h 3m 40s
"""blond""","""fair""","""blue""",19.0,1958-05-30,1h 58m 44s


In [22]:
rebels.select(cs.matches("^[a-z]{4}$"))

name,mass
str,f64
"""Han Solo""",80.0
"""Leia Organa""",49.0
"""Luke Skywalker""",77.0


### Selecting Based on Data Type

In [23]:
rebels.group_by("hair_color").agg(cs.numeric().mean())

hair_color,height,mass,birth_year
str,f64,f64,f64
"""blond""",172.0,77.0,19.0
"""brown""",165.0,64.5,24.0


In [24]:
rebels.select(cs.string())

name,hair_color,skin_color,eye_color,homeworld,species
str,str,str,str,str,str
"""Han Solo""","""brown""","""fair""","""brown""","""Corellia""","""Human"""
"""Leia Organa""","""brown""","""light""","""brown""","""Alderaan""","""Human"""
"""Luke Skywalker""","""blond""","""fair""","""blue""","""Tatooine""","""Human"""


In [25]:
rebels.select(cs.temporal())

birth_date,screen_time
date,duration[μs]
1948-06-01,1h 12m 37s
1958-05-30,1h 3m 40s
1958-05-30,1h 58m 44s


In [26]:
rebels.select(cs.by_dtype(pl.List(pl.String)))

vehicles,starships
list[str],list[str]
null,"[""Millennium Falcon"", ""Imperial shuttle""]"
"[""Imperial Speeder Bike""]",null
"[""Snowspeeder"", ""Imperial Speeder Bike""]","[""X-wing"", ""Imperial shuttle""]"


In [29]:
rebels.select(cs.list(cs.string()))

vehicles,starships
list[str],list[str]
null,"[""Millennium Falcon"", ""Imperial shuttle""]"
"[""Imperial Speeder Bike""]",null
"[""Snowspeeder"", ""Imperial Speeder Bike""]","[""X-wing"", ""Imperial shuttle""]"


### Selecting Based on Position

In [30]:
rebels.select(cs.by_index(range(0, 999, 3), require_all=False))

name,hair_color,birth_year,homeworld,starships
str,str,f64,str,list[str]
"""Han Solo""","""brown""",29.0,"""Corellia""","[""Millennium Falcon"", ""Imperial shuttle""]"
"""Leia Organa""","""brown""",19.0,"""Alderaan""",null
"""Luke Skywalker""","""blond""",19.0,"""Tatooine""","[""X-wing"", ""Imperial shuttle""]"


In [31]:
rebels.select("name", cs.by_index(range(-2, 0)))

name,birth_date,screen_time
str,date,duration[μs]
"""Han Solo""",1948-06-01,1h 12m 37s
"""Leia Organa""",1958-05-30,1h 3m 40s
"""Luke Skywalker""",1958-05-30,1h 58m 44s


In [ ]:
# This raises a ColumnNotFoundError:
# rebels.select(cs.by_index(20))

In [32]:
rebels.select(cs.by_index(range(20, 22), require_all=False))

shape: (0, 0)
┌┐
╞╡
└┘

### Combining Selectors

In [33]:
rebels.select(cs.by_name("hair_color") | cs.numeric())

height,mass,hair_color,birth_year
u16,f64,str,f64
180,80.0,"""brown""",29.0
150,49.0,"""brown""",19.0
172,77.0,"""blond""",19.0


In [37]:
df = pl.DataFrame({"d": 1, "i": True, "s": True, "c": True, "o": 1.0})

print(df)

x = cs.by_name("d", "i", "s")
y = cs.boolean()

print("\nselector => columns")

for s in ["x", "y", "x | y", "x & y", "x - y", "x ^ y", "~x", "x - x"]:
    print(f"{s:<8} => {cs.expand_selector(df, eval(s))}")

shape: (1, 5)
┌─────┬──────┬──────┬──────┬─────┐
│ d   ┆ i    ┆ s    ┆ c    ┆ o   │
│ --- ┆ ---  ┆ ---  ┆ ---  ┆ --- │
│ i64 ┆ bool ┆ bool ┆ bool ┆ f64 │
╞═════╪══════╪══════╪══════╪═════╡
│ 1   ┆ true ┆ true ┆ true ┆ 1.0 │
└─────┴──────┴──────┴──────┴─────┘

selector => columns
x        => ('d', 'i', 's')
y        => ('i', 's', 'c')
x | y    => ('d', 'i', 's', 'c')
x & y    => ('i', 's')
x - y    => ('d',)
x ^ y    => ('d', 'c')
~x       => ('c', 'o')
x - x    => ()


In [38]:
df.select(x - x)

shape: (0, 0)
┌┐
╞╡
└┘

In [39]:
print(df.select(first := cs.by_name("c", "i"), ~first))
print(f"first: {first}, ~first: {~first}")

shape: (1, 5)
┌──────┬──────┬─────┬──────┬─────┐
│ c    ┆ i    ┆ d   ┆ s    ┆ o   │
│ ---  ┆ ---  ┆ --- ┆ ---  ┆ --- │
│ bool ┆ bool ┆ i64 ┆ bool ┆ f64 │
╞══════╪══════╪═════╪══════╪═════╡
│ true ┆ true ┆ 1   ┆ true ┆ 1.0 │
└──────┴──────┴─────┴──────┴─────┘
first: cs.by_name('c', 'i', require_all=true), ~first: [cs.all() - cs.by_name('c', 'i', require_all=true)]


In [40]:
print(df.select(first := cs.last(), ~first))
print(f"first: {first}, ~first: {~first}")

shape: (1, 5)
┌─────┬─────┬──────┬──────┬──────┐
│ o   ┆ d   ┆ i    ┆ s    ┆ c    │
│ --- ┆ --- ┆ ---  ┆ ---  ┆ ---  │
│ f64 ┆ i64 ┆ bool ┆ bool ┆ bool │
╞═════╪═════╪══════╪══════╪══════╡
│ 1.0 ┆ 1   ┆ true ┆ true ┆ true │
└─────┴─────┴──────┴──────┴──────┘
first: cs.last(require=true), ~first: [cs.all() - cs.last(require=true)]


## Creating Columns

In [ ]:
rebels.with_columns(bmi=pl.col("mass") / ((pl.col("height") / 100) ** 2))

In [ ]:
df = pl.DataFrame({"a": [1, 2, 3]})
df.with_columns(pl.col("a") * 2)

In [ ]:
df.with_columns(a2=pl.col("a") * 2)

In [ ]:
rebels.with_columns(
    bmi=pl.col("mass") / ((pl.col("height") / 100) ** 2),
    age_destroy=(
        (pl.date(1983, 5, 25) - pl.col("birth_date")).dt.total_days() / 365
    ).cast(pl.UInt8),
)

In [ ]:
# This raises a ColumnNotFoundError:
# rebels.with_columns(
#     bmi=pl.col("mass") / ((pl.col("height") / 100) ** 2),
#     bmi_cat=pl.col("bmi").cut(
#         [18.5, 25], labels=["Underweight", "Normal", "Overweight"]
#     ),
# )

In [ ]:
(
    rebels.with_columns(
        bmi=pl.col("mass") / ((pl.col("height") / 100) ** 2)
    ).with_columns(
        bmi_cat=pl.col("bmi").cut(
            [18.5, 25], labels=["Underweight", "Normal", "Overweight"]
        )
    )
)

In [ ]:
# This raises a SyntaxError:
# starwars.select(
#     "name",
#     bmi=(pl.col("mass") / ((pl.col("height") / 100) ** 2)),
#     "species",
# )

In [ ]:
(
    starwars.select(
        "name",
        (pl.col("mass") / ((pl.col("height") / 100) ** 2)).alias("bmi"),  
        "species",
    )
    .drop_nulls()
    .top_k(5, by="bmi")  
)

In [ ]:
df.with_columns(pl.lit(1).alias("ones"))

In [ ]:
df.select(pl.all(), pl.lit(1).alias("ones"))

## Related Column Operations

### Dropping

In [ ]:
rebels.drop("name", "screen_time", strict=False)  

In [ ]:
rebels.select(~cs.by_name("name", "screen_time"))

In [ ]:
rebels.select(cs.exclude("name", "screen_time"))

### Renaming

In [ ]:
(
    rebels.rename({"homeworld": "planet", "mass": "weight"})
    .rename(lambda s: s.removesuffix("_color"))
    .select("name", "planet", "weight", "hair", "skin", "eye")  
)

### Stacking

In [ ]:
rebel_names = rebels.select("name")
rebel_colors = rebels.select(cs.ends_with("_color"))
rebel_quotes = pl.Series(
    "quote",
    [
        "You know, sometimes I amaze myself.",
        "That doesn't sound too hard.",
        "I have a bad feeling about this.",
    ],
)

(rebel_names.hstack(rebel_colors).hstack([rebel_quotes]))  

### Adding Row Indices

In [ ]:
rebels.with_row_index(name="rebel_id", offset=1)

## Takeaways